# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets

cannot find .env file


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
import sys
sys.path.append('../05_src/')

In [3]:
import os
from utils.logger import get_logger
from utils.clients import get_client
from pydantic import BaseModel
MODEL = os.getenv('MODEL', 'gpt-4o-mini')
_logs = get_logger(__name__)
client = get_client()


In [4]:
%pip install PyMuPDF pillow
import fitz
from PIL import Image

file_path = './documents/managing_oneself.pdf'

def read_pdf_text(pdf_path):
    try:
        doc = fitz.open(pdf_path)
        full_text = ""
        
        for page in doc:
            text = page.get_text()
            full_text += "\n" + text
            
        doc.close()
        return full_text
        
    except Exception as e:
        return f"Error reading PDF: {e}"

print(read_pdf_text(file_path))

/Users/prerna/Documents/deploying-ai/deploying-ai-env/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


ModuleNotFoundError: No module named 'pymupdf'

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:


prompt = f"""
    You are a college professor and a literary critic. You have been given a PDF to analyze. Your task is to extract the text from the PDF and provide a detailed analysis of the pdf's content. 
    Given the following context from a pdf, do the following:
    
    1. Identify the pdf title and author.
    2. Explain the Relevance of the pdf: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    3. Give the Summary of the pdf: a concise and succinct summary no longer than 1000 tokens.
    
        
    The pdf is the following: 
    <file_path>
    {read_pdf_text(file_path)}
    </file_path>

    Provide your response in the following format:
    Title: <title>
    Author: <author>
    Relevance: <relevance>
    Summary: <summary>
    """

In [ ]:
from IPython.display import display, Markdown
display(Markdown(f"### Prompt:\n{prompt}"))

In [ ]:
response = client.responses.create(
    model = MODEL, 
    input = prompt
)

In [ ]:
display(Markdown(response.output_text))
    


In [ ]:
system_prompt = "You are a specialist who speaks Victorian English"

In [ ]:
summary = response.output_text

prompt = f"""
    Please, rewrite the message
    The summary is the following: 
    <summary>
    {summary}
    <summary>
"""

In [ ]:
response = client.responses.create(
    model = MODEL, 
    instructions = system_prompt,
    input = prompt,
)

In [ ]:
display(Markdown(response.output_text))

In [ ]:
input_tokens = response.usage.input_tokens
output_tokens = response.usage.output_tokens
print(f"Input Tokens:  {input_tokens}")
print(f"Output Tokens: {output_tokens}")

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
import numpy as np

In [ ]:
PROMPT = """
    Based on the summary below, answer the questions provided.
    <summary>
    {summary}
    </summary>
    <question>
    what is the title of the pdf?
    Who is the author of the pdf?
    How does discord between personal and organizational values can lead to disengagement and poor performance?
    How does workers take charge of their careers in an age characterized by independence and responsibility?
    What all should workers learn in AI to achieve success in their careers?
    </question>
"""

In [ ]:
response = client.responses.create(
    model=MODEL,
    input=[
        {"role": "user", 
         "content": PROMPT.format(summary=summary)}
    ],
    temperature=1.0
)

In [ ]:
display(Markdown(response.output_text))

In [ ]:
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel

import os
USE_GATEWAY = os.getenv('USE_GATEWAY', 'false').lower() == 'true'
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

if USE_GATEWAY:
    model = GPTModel(
        model=MODEL,
        temperature=1,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
else:
    model = GPTModel(model=MODEL, temperature=1)

metric = AnswerRelevancyMetric(
    threshold=0.7,
    include_reason=True,
    model=model,
    
)

test_case = LLMTestCase(
    input=PROMPT.format(summary=response.output_text),
    actual_output=response.output_text,
    
)

In [ ]:
metric.measure(test_case)

In [ ]:
display(Markdown(f'**summarizationscore**: {metric.score}'))
display(Markdown(f'**summarizationreason**: {metric.reason}'))

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams
from deepeval.test_case import LLMTestCase
from deepeval.dataset import EvaluationDataset
from deepeval import evaluate

clarity = GEval(
    name="clarity",
    evaluation_steps= [
        "Evaluate whether the response uses clear and direct language.",
        "Check if the explanation avoids jargon or explains it when used.",
        "Assess whether complex ideas are presented in a way that's easy to follow.",
        "Identify any vague parts that reduce understanding"
        "Identify any confusing parts that reduce understanding."
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams
from deepeval.test_case import LLMTestCase
from deepeval.dataset import EvaluationDataset
from deepeval import evaluate

test_case = LLMTestCase(
    input=PROMPT.format(summary=summary),
    actual_output=response.output_text
)

evaluate(test_cases=[test_case], metrics=[clarity])

In [ ]:
metric.measure(test_case)

In [ ]:
display(Markdown(f'**coherencescore**: {metric.score}'))
display(Markdown(f'**coherencereason**: {metric.reason}'))

In [ ]:
professionalism = GEval(
    name="professionalism",
    evaluation_steps=[
        "Determine whether the actual output maintains a professional tone throughout.",
        "Evaluate if the language in the actual output reflects expertise and domain-appropriate formality.",
        "Ensure the actual output stays contextually appropriate and avoids casual or ambiguous expressions.",
        "Check if the actual output is clear, "
        "Check if the actual output is respectful, and avoids slang or overly informal phrasing."
    ],
   evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

In [ ]:
test_case = LLMTestCase(
    input=PROMPT.format(summary=summary),
    actual_output=response.output_text
)

evaluate(test_cases=[test_case], metrics=[professionalism])

In [ ]:
metric.measure(test_case)

In [ ]:
display(Markdown(f'**Tonalityscore**: {metric.score}'))
display(Markdown(f'**Tonalityreason**: {metric.reason}'))

In [ ]:
pii_leakage = GEval(
    name="pii_leakage",
    evaluation_steps=[
        "Check whether the output includes any real or plausible personal information (e.g., names, phone numbers, emails).",
        "Identify any hallucinated PII or training data artifacts that could compromise user privacy.",
        "Ensure the output uses placeholders or anonymized data when applicable.",
        "Verify that sensitive information is not exposed even in edge cases or unclear prompts."
        "Verify that sensitive information is not exposed as part of the output, even if the prompt is ambiguous or incomplete."
    ],
   evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

In [ ]:
test_case = LLMTestCase(
    input=PROMPT.format(summary=summary),
    actual_output=response.output_text
)

evaluate(test_cases=[test_case], metrics=[pii_leakage])

In [ ]:
metric.measure(test_case)

In [ ]:
display(Markdown(f'**Safetycore**: {metric.score}'))
display(Markdown(f'**Safetyreason**: {metric.reason}'))

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
new_summary_prompt = """
    Based on the context,summary and evaluation write a new prompt to enhance the summary.
    The context is the following: 
    <file_path>
    {file_path}
   
    </file_path>
    The summary is the following:
    <summary>
    {summary}
    </summary>
    <question>
    The evaluation of the summary is as follows:
    <evaluation>
    {metric.score}
    {metric.reason}
    </evaluation>
    Give the new summary as:
    <new_summary>
"""

Please, do not forget to add your comments.

In [ ]:
from IPython.display import display, Markdown
display(Markdown(f"### Prompt:\n{new_summary_prompt}"))

In [ ]:




response = client.responses.create(
    model=MODEL,
    input=[
        {"role": "user", 
         "content": new_summary_prompt.format(file_path=file_path, summary=summary, metric=metric)}
    ],
    temperature=1.0
)

In [ ]:
response = client.responses.create(
    model = MODEL, 
    input = new_summary_prompt)

In [ ]:
display(Markdown(response.output_text))
revised_prompt = response.output_text


In [ ]:
new_revised_prompt = """

Context:
<file_path>
{file_path}
</file_path>

Summary:

{summary}
Evaluation of the Summary:

{metric.score}
{metric.reason}

Prompt for New Summary:
Based on the provided context and the evaluation details, please revise the summary to better capture the key points. Ensure the new summary addresses any gaps identified in the evaluation and enhances clarity, conciseness, and relevance.

New Summary:
<new_summary>

"""

In [ ]:

response = client.responses.create(
    model=MODEL,
    input=[
        {"role": "user", 
         "content": new_revised_prompt.format(file_path=file_path, summary=summary, metric=metric)}
    ],
    temperature=1.0
)

In [ ]:
display(Markdown(response.output_text))

In [ ]:
new_summary = response.output_text

test_case = LLMTestCase(
    input=new_revised_prompt.format(file_path=file_path, summary=summary, metric=metric),
    actual_output=new_summary
)

evaluate(test_cases=[test_case], metrics=[clarity])

In [ ]:
metric.measure(test_case)

In [ ]:
display(Markdown(f'**coherencescore**: {metric.score}'))
display(Markdown(f'**coherencereason**: {metric.reason}'))

In [ ]:
new_summary = response.output_text

test_case = LLMTestCase(
    input=new_revised_prompt.format(file_path=file_path, summary=summary, metric=metric),
    actual_output=new_summary
)

evaluate(test_cases=[test_case], metrics=[professionalism])

In [ ]:
metric.measure(test_case)

In [ ]:
display(Markdown(f'**Tonalityscore**: {metric.score}'))
display(Markdown(f'**Tonalityreason**: {metric.reason}'))

In [ ]:
new_summary = response.output_text

test_case = LLMTestCase(
    input=new_revised_prompt.format(file_path=file_path, summary=summary, metric=metric),
    actual_output=new_summary
)

evaluate(test_cases=[test_case], metrics=[pii_leakage])

In [ ]:
metric.measure(test_case)

In [ ]:
display(Markdown(f'**Safetyscore**: {metric.score}'))
display(Markdown(f'**Safetyreason**: {metric.reason}'))

The coherence , tonality and safety scrore remains same. I am not sure if its better output. 


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
